# Allegiant Air — Per-Subject CCPA/DSAR Erasure (POC)  ·  0. Setup & generate

**What this solution is.** A OneTrust/CCPA/DSAR request names **one specific customer** who wants their
data deleted. This job erases **only that subject's rows** across every registered table — *in place, in the
same table* — while **every other customer's value in the same column stays intact**, then **physically
purges** the raw bytes (CCPA "no trace").

**How it differs from the masking repo (kept, unchanged).** The `pii_masking_json_demo` notebook does
*blanket, day-to-day column masking* (mask a whole key/column for everyone, via governed tag + ABAC). That is
the wrong shape for erasure — it would blank the column for all customers. This is the **targeted, per-subject,
materialized-and-purged** layer built alongside it.

**This notebook builds a fully self-contained demo on a NEW schema and NEW tables** (nothing here touches the
existing `allegiant_air` masking demo):
- `customer_profile` — scalar `first_name` / `last_name` / `email` (+ non-PII revenue) → *all-three-match* case
- `contact_email_only` — only `email` (+ phone) → *table missing some identifiers* case
- `booking` — single `full_name` column + `pnr` → *single-name + PNR* case
- `loyalty_split` — split `first_nm` / `last_nm` → *split-name* case
- `app_hits_json` — GA-style nested **JSON** (name key + URL-embedded first/last/email) → *in-JSON* case
- `dsar_request` — the request table (replaces DynamoDB): subject name/email + request_type + 45-day deadline + status

**Idempotent:** re-running drops & recreates the DSAR schema (a clean rebuild every time), so it never
collides with prior runs. Erasure value for scalars = a fixed **redaction token** (`***REDACTED***`); JSON
values get **targeted redaction**. Row match = **all registered identifiers present on that table must match**
(most conservative).

## 0. Configuration

In [3]:
import json

dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema (separate from masking demo)")
dbutils.widgets.text("num_background_rows", "5000", "3 Background (non-target) rows per table")
dbutils.widgets.text("redaction_token", "***REDACTED***", "4 Scalar erasure token")
dbutils.widgets.text("deadline_days", "45", "5 Deadline days from request date")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
NBG     = int(dbutils.widgets.get("num_background_rows"))
TOKEN   = dbutils.widgets.get("redaction_token")
DDAYS   = int(dbutils.widgets.get("deadline_days"))
FQ      = f"{CATALOG}.{SCHEMA}"
print("Target schema:", FQ)

Target schema: dkushari_uc.allegiant_air_dsar

## 1. Clean rebuild of the DSAR schema
Drop & recreate so every run is deterministic and isolated from the masking demo's `allegiant_air` schema.

In [5]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"DROP SCHEMA IF EXISTS {FQ} CASCADE")
spark.sql(f"CREATE SCHEMA {FQ} COMMENT 'Allegiant per-subject CCPA/DSAR erasure POC — self-contained, isolated from the blanket-masking demo.'")
print("Recreated", FQ)

Recreated dkushari_uc.allegiant_air_dsar

## 2. The 10 DSAR subjects (this month's OneTrust requests)
These are the specific customers who asked for erasure. We deliberately plant their rows in the tables below
(alongside thousands of other customers) so the demo can prove: **only these subjects' rows are erased.**

In [7]:
# (first_name, last_name, email, request_type)   — request_type: DELETE or OBFUSCATE (treated identically here)
subjects = [
    ("Andra",   "Mayfield",  "andra.mayfield@example.com",   "DELETE"),
    ("Richard", "Jones",     "richard.jones@example.com",    "OBFUSCATE"),
    ("Maria",   "Gonzalez",  "maria.gonzalez@example.com",   "DELETE"),
    ("Wei",     "Chen",      "wei.chen@example.com",         "DELETE"),
    ("Fatima",  "Al-Sayed",  "fatima.alsayed@example.com",   "OBFUSCATE"),
    ("James",   "O'Brien",   "james.obrien@example.com",     "DELETE"),
    ("Priya",   "Nair",      "priya.nair@example.com",       "DELETE"),
    ("Diego",   "Santos",    "diego.santos@example.com",     "OBFUSCATE"),
    ("Emily",   "Nguyen",    "emily.nguyen@example.com",     "DELETE"),
    ("Tomas",   "Novak",     "tomas.novak@example.com",      "DELETE"),
]
print(len(subjects), "subjects")

def sqlstr(s):
    return "'" + s.replace("'", "''") + "'"

subjects_values = ",\n".join(
    f"({sqlstr(fn)}, {sqlstr(ln)}, {sqlstr(em)}, {sqlstr(rt)})"
    for fn, ln, em, rt in subjects
)

10 subjects

## 3. `dsar_request` — the request table (replaces DynamoDB)
One row per privacy request: subject identifiers + type + request_date + **deadline (request_date + N days)** +
`status` (PENDING). The engine reads PENDING rows; the purge step later scrubs the raw identifiers here too.

In [9]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.dsar_request (
  request_id    STRING,
  first_name    STRING,
  last_name     STRING,
  email         STRING,
  request_type  STRING,
  request_date  DATE,
  deadline_date DATE,
  status        STRING
) COMMENT 'DSAR/CCPA privacy requests (native replacement for the DynamoDB request table).'
""")

spark.sql(f"""
INSERT INTO {FQ}.dsar_request
SELECT
  concat('REQ-', lpad(cast(row_number() over (order by email) as string), 4, '0')) AS request_id,
  first_name, last_name, email, request_type,
  current_date() AS request_date,
  date_add(current_date(), {DDAYS}) AS deadline_date,
  'PENDING' AS status
FROM (VALUES
{subjects_values}
) AS t(first_name, last_name, email, request_type)
""")
display(spark.table(f"{FQ}.dsar_request"))

request_id,first_name,last_name,email,request_type,request_date,deadline_date,status
REQ-0001,Andra,Mayfield,andra.mayfield@example.com,DELETE,2026-07-23,2026-09-06,PENDING
REQ-0002,Diego,Santos,diego.santos@example.com,OBFUSCATE,2026-07-23,2026-09-06,PENDING
REQ-0003,Emily,Nguyen,emily.nguyen@example.com,DELETE,2026-07-23,2026-09-06,PENDING
REQ-0004,Fatima,Al-Sayed,fatima.alsayed@example.com,OBFUSCATE,2026-07-23,2026-09-06,PENDING
REQ-0005,James,OBrien,james.obrien@example.com,DELETE,2026-07-23,2026-09-06,PENDING
REQ-0006,Maria,Gonzalez,maria.gonzalez@example.com,DELETE,2026-07-23,2026-09-06,PENDING
REQ-0007,Priya,Nair,priya.nair@example.com,DELETE,2026-07-23,2026-09-06,PENDING
REQ-0008,Richard,Jones,richard.jones@example.com,OBFUSCATE,2026-07-23,2026-09-06,PENDING
REQ-0009,Tomas,Novak,tomas.novak@example.com,DELETE,2026-07-23,2026-09-06,PENDING
REQ-0010,Wei,Chen,wei.chen@example.com,DELETE,2026-07-23,2026-09-06,PENDING


## 4. Background customers (the "everyone else" whose data must survive)
A reusable temp view of synthetic non-target people. None of them share a name/email with the 10 subjects.

In [11]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW bg AS
SELECT
  id,
  element_at(array('Liam','Olivia','Noah','Emma','Oliver','Ava','Elijah','Sophia','Lucas','Mia',
                   'Mason','Isabella','Ethan','Amelia','Logan','Harper','Jack','Evelyn','Aiden','Abigail'),
             cast(pmod(hash(id, 1), 20) as int) + 1) AS first_name,
  element_at(array('Smith','Brown','Wilson','Taylor','Davies','Evans','Thomas','Roberts','Walker','Wright',
                   'Green','Hall','Wood','Harris','Martin','Clarke','Lewis','Young','King','Scott'),
             cast(pmod(hash(id, 2), 20) as int) + 1) AS last_name,
  concat('user', cast(id as string), '@example.com') AS email,
  cast(50 + pmod(hash(id, 3), 950) as int) AS amount
FROM range({NBG}) AS r(id)
""")
print("background rows:", spark.table("bg").count())

background rows: 5000

## 5a. `customer_profile` — scalar first/last/email (+ non-PII revenue)
The clean *all-three-match* case. Each subject gets one row here; the rest are background customers.

In [13]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.customer_profile (
  customer_id  BIGINT,
  first_name   STRING,
  last_name    STRING,
  email        STRING,
  home_city    STRING,
  lifetime_revenue INT
) COMMENT 'Scalar PII: first_name/last_name/email. Non-PII: home_city, lifetime_revenue (must survive erasure).'
""")
# background
spark.sql(f"""
INSERT INTO {FQ}.customer_profile
SELECT id, first_name, last_name, email,
       element_at(array('Las Vegas','Phoenix','Orlando','Austin','Denver'), cast(pmod(hash(id,7),5) as int)+1),
       amount FROM bg
""")
# the 10 subjects (offset ids so they are distinct)
spark.sql(f"""
INSERT INTO {FQ}.customer_profile
SELECT 1000000 + row_number() over (order by email) AS customer_id,
       first_name, last_name, email, 'Las Vegas', 4200
FROM (VALUES
{subjects_values}
) AS t(first_name, last_name, email, request_type)
""")
print("customer_profile:", spark.table(f"{FQ}.customer_profile").count())

customer_profile: 5010

## 5b. `contact_email_only` — only email present (missing-identifier case)
Kartik's real example: a table where you can only match on **email**. Registry will register email as the
single identifier, so the "all registered identifiers must match" rule reduces to email-only here.

In [15]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.contact_email_only (
  contact_id BIGINT,
  email      STRING,
  phone      STRING,
  opt_in     BOOLEAN
) COMMENT 'Only email is PII/identifiable here. phone/opt_in non-PII. Demonstrates email-only targeted match.'
""")
spark.sql(f"""
INSERT INTO {FQ}.contact_email_only
SELECT id, email, concat('+1-702-', lpad(cast(pmod(hash(id,9),1000) as string),3,'0'), '-',
       lpad(cast(pmod(hash(id,11),10000) as string),4,'0')), (pmod(id,2)=0) FROM bg
""")
spark.sql(f"""
INSERT INTO {FQ}.contact_email_only
SELECT 2000000 + row_number() over (order by email), email, '+1-702-555-0100', true
FROM (VALUES
{subjects_values}
) AS t(first_name, last_name, email, request_type)
""")
print("contact_email_only:", spark.table(f"{FQ}.contact_email_only").count())

contact_email_only: 5010

## 5c. `booking` — single `full_name` column + `pnr`
The single-name-column + PNR case. Match is on the whole name string `full_name = 'First Last'`.
`pnr` is also PII (a booking reference tied to the person) and is erased for matched rows.

In [17]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.booking (
  booking_id BIGINT,
  full_name  STRING,
  pnr        STRING,
  flight_no  STRING,
  fare_usd   INT
) COMMENT 'PII: full_name (single column) + pnr. Non-PII: flight_no, fare_usd (must survive).'
""")
spark.sql(f"""
INSERT INTO {FQ}.booking
SELECT id, concat(first_name,' ',last_name),
       upper(concat(substr(md5(cast(id as string)),1,6))),
       concat('G4', cast(100 + pmod(hash(id,13),900) as string)),
       amount FROM bg
""")
spark.sql(f"""
INSERT INTO {FQ}.booking
SELECT 3000000 + row_number() over (order by email), concat(first_name,' ',last_name),
       upper(concat('PNR', lpad(cast(row_number() over (order by email) as string),3,'0'))),
       'G4777', 512
FROM (VALUES
{subjects_values}
) AS t(first_name, last_name, email, request_type)
""")
print("booking:", spark.table(f"{FQ}.booking").count())

booking: 5010

## 5d. `loyalty_split` — split `first_nm` / `last_nm`
Split-name columns (differently named than customer_profile on purpose — the registry maps physical column
names to logical identifiers). Match requires **both** first_nm AND last_nm.

In [19]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.loyalty_split (
  member_id BIGINT,
  first_nm  STRING,
  last_nm   STRING,
  tier      STRING,
  points    INT
) COMMENT 'PII: first_nm/last_nm. Non-PII: tier, points (must survive).'
""")
spark.sql(f"""
INSERT INTO {FQ}.loyalty_split
SELECT id, first_name, last_name,
       element_at(array('SILVER','GOLD','PLATINUM'), cast(pmod(hash(id,17),3) as int)+1),
       cast(pmod(hash(id,19),50000) as int) FROM bg
""")
spark.sql(f"""
INSERT INTO {FQ}.loyalty_split
SELECT 4000000 + row_number() over (order by email), first_name, last_name, 'GOLD', 25000
FROM (VALUES
{subjects_values}
) AS t(first_name, last_name, email, request_type)
""")
print("loyalty_split:", spark.table(f"{FQ}.loyalty_split").count())

loyalty_split: 5010

## 5e. `app_hits_json` — PII **inside** nested JSON
GA-style event blob: `appInfo.name` = the person, and a `manage-travel` URL carrying
`firstName`/`lastName`/`email`. `appName` (`abc2Go`) and metrics are non-PII and must survive.
This is where the erasure engine reuses a `regexp_replace` mask (targeted at the matched subject's row).

In [21]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.app_hits_json (
  hit_id      BIGINT,
  hit_payload STRING,
  revenue_usd INT
) COMMENT 'PII lives INSIDE hit_payload JSON (name key + URL firstName/lastName/email). appName + revenue non-PII.'
""")
# background hits (random non-target people)
spark.sql(f"""
INSERT INTO {FQ}.app_hits_json
SELECT id,
  to_json(named_struct(
    'appInfo', named_struct('appId','com.lixar.abc','appName','abc2Go','name', concat(first_name,' ',last_name)),
    'documentLocation', concat('https://www.allegiant.com/manage-travel?firstName=', first_name,
                               '&lastName=', last_name, '&email=', email, '&pnr=AB', cast(pmod(id,99) as string)),
    'metrics', named_struct('revenue', amount)
  )),
  amount FROM bg
""")
# subject hits
spark.sql(f"""
INSERT INTO {FQ}.app_hits_json
SELECT 5000000 + row_number() over (order by email),
  to_json(named_struct(
    'appInfo', named_struct('appId','com.lixar.abc','appName','abc2Go','name', concat(first_name,' ',last_name)),
    'documentLocation', concat('https://www.allegiant.com/manage-travel?firstName=', first_name,
                               '&lastName=', last_name, '&email=', email, '&pnr=ZZ999'),
    'metrics', named_struct('revenue', 999)
  )),
  999
FROM (VALUES
{subjects_values}
) AS t(first_name, last_name, email, request_type)
""")
print("app_hits_json:", spark.table(f"{FQ}.app_hits_json").count())
display(spark.sql(f"SELECT * FROM {FQ}.app_hits_json WHERE revenue_usd = 999 LIMIT 3"))

hit_id,hit_payload,revenue_usd
230,"{""appInfo"":{""appId"":""com.lixar.abc"",""appName"":""abc2Go"",""name"":""Liam Wright""},""documentLocation"":""https://www.allegiant.com/manage-travel?firstName=Liam&lastName=Wright&email=user230@example.com&pnr=AB32"",""metrics"":{""revenue"":999}}",999
402,"{""appInfo"":{""appId"":""com.lixar.abc"",""appName"":""abc2Go"",""name"":""Noah Wright""},""documentLocation"":""https://www.allegiant.com/manage-travel?firstName=Noah&lastName=Wright&email=user402@example.com&pnr=AB6"",""metrics"":{""revenue"":999}}",999
2924,"{""appInfo"":{""appId"":""com.lixar.abc"",""appName"":""abc2Go"",""name"":""Jack Harris""},""documentLocation"":""https://www.allegiant.com/manage-travel?firstName=Jack&lastName=Harris&email=user2924@example.com&pnr=AB53"",""metrics"":{""revenue"":999}}",999


## 6. Summary
All demo tables built on the isolated `allegiant_air_dsar` schema. Next: **`01_pii_column_registry`** to
register which columns are PII and how each is erased, then **`02_subject_erasure_engine`** to erase only
the 10 subjects.

In [23]:
for t in ["dsar_request","customer_profile","contact_email_only","booking","loyalty_split","app_hits_json"]:
    print(f"{t:20s}", spark.table(f"{FQ}.{t}").count(), "rows")

dsar_request         10 rows
customer_profile     5010 rows
contact_email_only   5010 rows
booking              5010 rows
loyalty_split        5010 rows
app_hits_json        5010 rows